# Instruction Fine-Tuning

**Turning a raw pretrained language model into one that reliably follows natural-language instructions — by supervised training on curated `(instruction, response)` pairs.**

Instruction fine-tuning (IFT), also called supervised fine-tuning (SFT), is the first alignment stage that sits between *pretraining* (predict the next token over web-scale text) and *preference optimization* (RLHF/DPO). A base model is a brilliant autocomplete that does not know it is supposed to *answer* you; IFT teaches it the conversational contract — read an instruction, produce a helpful, correctly formatted response — and is the single cheapest, highest-leverage way to adapt an open model to a task, domain, format, or persona.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

### What is it?

**Instruction fine-tuning** is supervised learning on a dataset of `(prompt, response)` examples where the prompt is a natural-language instruction (optionally with input/context) and the response is the desired output. The model is trained with the ordinary causal-language-modeling loss (cross-entropy over next tokens) — the only twist is that the loss is usually **masked over the prompt tokens** so the model is graded only on *generating* the response, not on echoing the question.

The standard pipeline is:

1. **Pretraining** — next-token prediction over trillions of tokens → a *base* model.
2. **Instruction fine-tuning (this notebook)** → an *instruct* / *chat* model that follows directions.
3. **Preference optimization** (RLHF, DPO, ORPO) → polish helpfulness, harmlessness, and tone.

IFT is what converts `Llama-3.1-8B` into `Llama-3.1-8B-Instruct`, and what you run yourself when you want a model specialized to your domain, format, or company voice.

### Why use it?

- **Instruction following** — base models complete text; they don't reliably answer questions, follow constraints, or stop. IFT installs that behavior.
- **Format & schema control** — teach the model to always emit valid JSON, a specific tool-call format, Markdown tables, or a fixed report structure.
- **Domain & style adaptation** — bake in legal, medical, or code-domain conventions and a consistent persona/voice that prompting alone struggles to hold.
- **Cost** — orders of magnitude cheaper than pretraining; with parameter-efficient methods (LoRA/QLoRA) you can fine-tune a 7–8B model on a single consumer/cloud GPU in hours.
- **Latency & cost at inference** — a small fine-tuned model can match a much larger prompted model on a narrow task, shrinking serving cost and prompt length.

### When to use it?

- You need **consistent behavior or output format** that few-shot prompting can't reliably enforce.
- You have (or can build) **hundreds-to-thousands of high-quality examples** of the task.
- You want a **smaller, cheaper, faster** model to replace expensive prompting of a frontier model.
- The task needs **style, tone, or domain conventions** that live in examples, not in a knowledge base.

Reach for **RAG instead** when the gap is missing *facts/knowledge* (retrieval fixes that); reach for **prompting instead** when the task is rare, changes constantly, or you have too few examples.

## Key Features

### Core Capabilities of Instruction Fine-Tuning

| Feature | Description | Benefit |
|---------|-------------|---------|
| Prompt-token loss masking | Compute loss only on response tokens (label `-100` on the prompt) | Model learns to *answer*, not to parrot the question |
| Chat templates | Apply the model's exact special-token format (`tokenizer.apply_chat_template`) | Train/serve parity; no silent format drift |
| Parameter-efficient tuning (LoRA/QLoRA) | Train small low-rank adapters, optionally over a 4-bit base | Fine-tune 7–70B models on 1 GPU; tiny, swappable artifacts |
| Sequence packing | Concatenate short examples to fill the context window | 2–5x higher GPU throughput, no wasted padding |
| Mixed precision + flash attention | bf16/fp16 compute with fused attention kernels | Faster steps, lower memory, longer contexts |
| Gradient checkpointing & accumulation | Trade compute for memory; simulate large batches | Train bigger models / batches than VRAM allows |
| Reproducible data recipe | Versioned dataset + template + hyperparameters | Auditable, repeatable runs and safe rollbacks |

## Architecture Overview

The instruction fine-tuning pipeline, from raw examples to a served model:

```
  +------------------+     +-----------------------+     +----------------------+
  | Instruction data | --> | Format & templating   | --> | Tokenize + label     |
  | (prompt,response)|     | apply_chat_template() |     | mask (prompt = -100) |
  +------------------+     +-----------------------+     +----------+-----------+
                                                                    |
                                                                    v
  +------------------+     +-----------------------+     +----------------------+
  | Base model       | --> | Trainer               | <-- | Packed batches       |
  | (HF / local)     |     |  - full FT  OR        |     | bf16, flash-attn,    |
  | optional 4-bit   |     |  - LoRA / QLoRA (PEFT) |     | grad checkpoint/accum|
  +------------------+     +-----------+-----------+     +----------------------+
                                       |
                    +------------------+------------------+
                    v                                     v
          +-------------------+                 +--------------------------+
          | Evaluation        |                 | Adapter / merged weights |
          | held-out loss +   |                 | merge_and_unload() OR    |
          | task benchmarks   |                 | load adapter at serve    |
          +-------------------+                 +------------+-------------+
                                                             |
                                                             v
                                                   +--------------------+
                                                   | Inference runtime  |
                                                   | (vLLM/TGI/Triton)  |
                                                   +--------------------+
```

### Components

1. **Instruction dataset** — `(instruction, optional input, response)` triples, often plus a `system` prompt. Quality and diversity matter far more than raw count.
2. **Formatting / chat template** — render each example into the model's exact token format. Always use the tokenizer's own chat template rather than hand-built strings.
3. **Tokenization + label masking** — convert to token ids and set prompt-token labels to `-100` so loss is computed only on the response.
4. **Base model** — a pretrained checkpoint, optionally quantized to 4-bit (QLoRA) to fit in memory.
5. **Trainer** — full fine-tuning (update all weights) or PEFT (train LoRA adapters). TRL's `SFTTrainer` wraps the Hugging Face `Trainer` with SFT conveniences (packing, masking, template handling).
6. **Evaluation** — held-out loss plus *task-level* benchmarks; loss alone does not measure instruction-following quality.
7. **Export & serve** — keep the adapter separate (swap at serve time) or `merge_and_unload()` into a standalone model for runtimes like vLLM/TGI.

## Installation

### Prerequisites

- Python 3.9+
- An NVIDIA GPU with CUDA 12.x for anything beyond toy models (QLoRA of a 7–8B model needs ~12–16 GB VRAM; full fine-tuning needs far more)
- A base model you are licensed to use (e.g. from the Hugging Face Hub) and a `HF_TOKEN` for gated models
- An instruction dataset in a `(prompt, response)` or chat-messages schema

### Installation Steps

The core stack is Hugging Face `transformers` + `datasets`, `trl` for the `SFTTrainer`, `peft` for LoRA, and `bitsandbytes` for 4-bit QLoRA. **Uncomment in Colab** or run in a virtualenv.

In [ ]:
# Uncomment the lines you need.

# Core SFT stack (Transformers + TRL trainer + PEFT/LoRA + datasets):
# !pip install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "datasets>=2.20" accelerate

# 4-bit QLoRA support (fits large models on a single GPU):
# !pip install -U bitsandbytes

# Faster attention kernels (optional, requires a compatible GPU + build toolchain):
# !pip install flash-attn --no-build-isolation

# For multi-GPU / large-scale full fine-tuning, configure accelerate or deepspeed:
# !pip install deepspeed && accelerate config

## Basic Usage

### Quick Start Example

The two ideas you must get right before any training are (1) **rendering each example with the model's chat template** and (2) **masking the loss to the response only**. The cell below builds a tiny instruction dataset and shows the exact formatted string a model would be trained on — this is the part people most often get wrong, and it requires no GPU to reason about.

In [ ]:
# Format instruction examples the way a chat model expects.
# This runs with no model weights — it only demonstrates the data contract.

raw_examples = [
    {
        "system": "You are a concise assistant.",
        "instruction": "Convert 3 miles to kilometers.",
        "response": "3 miles is about 4.83 km.",
    },
    {
        "system": "You are a concise assistant.",
        "instruction": "Name the capital of Japan.",
        "response": "Tokyo.",
    },
]

# In real code you would use the model's own template:
#   from transformers import AutoTokenizer
#   tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
#   text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
# Here is a faithful, illustrative version of a ChatML-style template:

def render(example):
    return (
        f"<|system|>\n{example['system']}<|end|>\n"
        f"<|user|>\n{example['instruction']}<|end|>\n"
        f"<|assistant|>\n{example['response']}<|end|>\n"
    )

def to_messages(example):
    return [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]

print(render(raw_examples[0]))
print("messages schema (what apply_chat_template consumes):")
print(to_messages(raw_examples[0]))


### Loss masking: why the prompt is labelled `-100`

During SFT we want the model graded only on the *response* tokens. The convention in PyTorch / Hugging Face is to set the label of every token we don't want to learn to `-100`, which `CrossEntropyLoss(ignore_index=-100)` skips. The cell below shows the label vector for one example so the concept is concrete.

In [ ]:
# Illustrate prompt-vs-response loss masking with a toy "tokenizer" (whitespace split).

prompt = "<user> Convert 3 miles to kilometers. <assistant>"
response = "3 miles is about 4.83 km."

prompt_tokens = prompt.split()
response_tokens = response.split()
tokens = prompt_tokens + response_tokens

# -100 = ignored by the loss; real ids would be the response token ids.
IGNORE = -100
labels = [IGNORE] * len(prompt_tokens) + list(range(len(response_tokens)))

for tok, lab in zip(tokens, labels):
    role = "PROMPT (masked)" if lab == IGNORE else "RESPONSE (learned)"
    print(f"{tok:<12} label={lab!s:<5} -> {role}")

n_learned = sum(1 for l in labels if l != IGNORE)
print(f"\nLoss is computed on {n_learned}/{len(labels)} tokens (response only).")


### The training call

With the data formatted and masking understood, the actual training loop is a few lines via TRL's `SFTTrainer`, which handles templating, packing, and masking for you. This is a reference snippet (it downloads weights and needs a GPU, so it is left commented).

In [ ]:
# Reference: a minimal full or LoRA SFT run with TRL's SFTTrainer.
# Requires a GPU + `pip install trl peft datasets`. Shown commented.
#
# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from trl import SFTConfig, SFTTrainer
#
# model_id = "meta-llama/Llama-3.1-8B-Instruct"
# tok = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="bfloat16")
#
# # Dataset rows must contain a "messages" column (list of role/content dicts);
# # SFTTrainer applies the chat template and masks the prompt automatically.
# ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:2000]")
#
# cfg = SFTConfig(
#     output_dir="out/llama-sft",
#     num_train_epochs=2,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=8,     # effective batch = 32
#     learning_rate=2e-5,
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.03,
#     bf16=True,
#     packing=True,                      # pack short samples to fill the context
#     max_seq_length=4096,
#     gradient_checkpointing=True,
#     logging_steps=10,
#     eval_strategy="steps", eval_steps=100,
#     save_strategy="steps", save_steps=200,
# )
#
# trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok)
# trainer.train()
# trainer.save_model("out/llama-sft")
print("SFTTrainer reference: full fine-tuning recipe (uncomment with a GPU to run)")


## Advanced Features

### LoRA / QLoRA, packing, and chat templates

#### LoRA (Low-Rank Adaptation)

Instead of updating all weights, LoRA freezes the base model and injects small trainable rank-`r` matrices into selected projection layers (typically the attention `q/k/v/o` and the MLP). Only ~0.1–1% of parameters train, so memory and checkpoint size collapse, and you get a tiny portable **adapter** you can swap or stack.

#### QLoRA

QLoRA loads the **base model in 4-bit** (NF4 quantization) and trains LoRA adapters on top in bf16. This is what lets a 7–13B (even 70B) model fine-tune on a single GPU. Quality is typically within a point or two of full fine-tuning for most adaptation tasks.

#### Packing

Most instruction examples are far shorter than the context window, so naive batching wastes GPU on padding. **Packing** concatenates multiple examples into one full-length sequence (with attention boundaries respected), often a 2–5x throughput win.

#### Chat templates

Every instruct model has a *specific* special-token format. Always call `tokenizer.apply_chat_template(...)` rather than hand-writing prompt strings — a mismatched template at train or serve time silently wrecks quality.

The cell below is the LoRA/QLoRA configuration you would pass to the trainer.

In [ ]:
# Reference: QLoRA configuration (4-bit base + LoRA adapters).
# Requires GPU + `pip install peft bitsandbytes`. Shown commented.
#
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
#
# bnb = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",            # normal-float-4, best for QLoRA
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
# base = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.1-8B-Instruct",
#     quantization_config=bnb,
#     device_map="auto",
# )
# base = prepare_model_for_kbit_training(base)   # enables grad checkpointing, casts norms
#
# lora = LoraConfig(
#     r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
#                     "gate_proj", "up_proj", "down_proj"],
# )
# model = get_peft_model(base, lora)
# model.print_trainable_parameters()    # e.g. "trainable: 0.62% of all params"
#
# # After training, fold the adapter into the base for single-artifact serving:
# # merged = model.merge_and_unload(); merged.save_pretrained("out/merged")
print("QLoRA reference: 4-bit NF4 base + rank-16 LoRA adapters on attention+MLP")


## Use Cases

### Real-world Applications

#### Use Case 1: Structured extraction (always-valid JSON)

- **Context** — an ingestion pipeline must turn free-text invoices into a strict JSON schema; a prompted general model emits valid JSON ~95% of the time, and the 5% failures break downstream parsing.
- **Implementation** — QLoRA fine-tune a 7B model on a few thousand `(document, JSON)` pairs, with the target serialized exactly to the schema. Enforce with constrained decoding at serve time.
- **Results** — near-100% schema-valid output, shorter prompts (no few-shot examples needed), and a much cheaper model than the frontier one it replaced.

#### Use Case 2: Domain assistant with a fixed voice

- **Context** — an internal support bot must answer in the company's tone and terminology, deferring to policy.
- **Implementation** — SFT on curated `(question, on-brand answer)` pairs plus a consistent system prompt; pair with RAG for live facts so the model supplies *style and behavior*, retrieval supplies *knowledge*.
- **Results** — consistent persona and formatting that prompting could not hold, with up-to-date facts from retrieval.

#### Use Case 3: Tool-calling / function-calling format

- **Context** — an agent needs the model to emit tool calls in an exact JSON function-call format reliably.
- **Implementation** — fine-tune on traces of `(user request, tool-call sequence, final answer)` so the format and decision of *when* to call a tool are learned.
- **Results** — higher tool-call validity and fewer malformed calls than prompt-only, enabling a smaller model in the agent loop.

## Best Practices

1. **Data quality over quantity** — a few thousand clean, diverse, correctly formatted examples beat hundreds of thousands of noisy ones. Deduplicate, filter, and manually review a sample.
2. **Always use the model's chat template** — render with `apply_chat_template` for both training and serving; a template mismatch is the most common silent quality killer.
3. **Mask the prompt** — compute loss only on response tokens (`-100` labels). Training on the prompt teaches the model to parrot questions.
4. **Start with QLoRA** — it is cheap, fast, and a strong baseline; only move to full fine-tuning if you have measured a clear quality gap and the hardware to pay for it.
5. **Hold out a real eval set** — keep validation data the model never trains on, and evaluate *task quality* (exact-match, schema-validity, win-rate), not just loss.
6. **Tune conservatively** — 1–3 epochs, LR ~1e-5–2e-5 for full FT (1e-4–2e-4 for LoRA), cosine schedule with warmup. More epochs usually means overfitting/forgetting, not better.
7. **Watch for catastrophic forgetting** — narrow fine-tuning can erode general ability; mix in some general instruction data or keep LoRA `r` modest to preserve the base.
8. **Version everything** — dataset hash, base model, template, and hyperparameters. Reproducibility and rollback depend on it.
9. **Set `eos`/stop correctly** — ensure the template emits the end-of-turn token so the model learns to *stop*; a missing EOS produces models that ramble forever.

## Common Pitfalls

1. **Wrong / hand-built prompt template** — manually concatenating strings that don't match the tokenizer's special tokens. *Avoid:* always use `apply_chat_template`; verify the rendered string includes the right BOS/EOS/role markers.
2. **Not masking the prompt** — computing loss over the instruction makes the model echo questions and degrades answers. *Avoid:* set prompt labels to `-100` (TRL's `SFTTrainer` with a chat dataset does this for you).
3. **Too many epochs / LR too high** — the model memorizes the training set and loses general ability (overfitting + catastrophic forgetting). *Avoid:* 1–3 epochs, low LR, early-stop on held-out loss, mix in general data.
4. **Evaluating on loss alone** — loss can drop while instruction-following gets worse. *Avoid:* measure task-level metrics and run qualitative spot checks.
5. **Train/serve skew** — a different template, system prompt, or quantization at inference than at training. *Avoid:* serve with the exact template used in training; test the merged/adapter model before rollout.
6. **Forgetting the EOS token** — examples without an end-of-turn marker yield models that never stop. *Avoid:* confirm the template adds EOS and that it is *learned* (not masked).
7. **Imbalanced or narrow data** — overrepresenting one task/format biases the model. *Avoid:* balance and diversify; cap dominant categories.

## Performance Optimization

### Configuration Tuning

Key levers, from highest to lowest impact for most SFT runs:

- **QLoRA vs full FT** — 4-bit base + LoRA cuts memory ~4x+ and trains far fewer parameters; the biggest single cost lever.
- **Sequence packing** — fill the context with concatenated examples to eliminate padding waste (often 2–5x throughput).
- **Mixed precision (bf16) + flash attention** — faster matmuls and fused attention with lower memory; enables longer `max_seq_length`.
- **Gradient checkpointing + accumulation** — fit larger models/batches by trading compute for memory and simulating big effective batch sizes.
- **Batch size & effective batch** — `per_device_batch_size * grad_accum * num_gpus`; raise the effective batch for stabler training up to the memory ceiling.
- **`max_seq_length`** — set to the real data distribution; over-long contexts waste compute, too-short truncates responses.

The cell below estimates a LoRA run's trainable-parameter savings and effective batch size — quick sanity math to run before committing GPU hours.

In [ ]:
# Quick planning math: LoRA trainable-parameter savings + effective batch size.

def lora_trainable_params(hidden=4096, n_layers=32, r=16, n_proj=7):
    """Approximate LoRA params: each targeted projection adds 2 * hidden * r."""
    per_layer = n_proj * (2 * hidden * r)
    return per_layer * n_layers

def full_params(hidden=4096, n_layers=32):
    # Very rough: attention (4*h^2) + MLP (~3 * h * 4h) per layer.
    per_layer = 4 * hidden**2 + 3 * hidden * (4 * hidden)
    return per_layer * n_layers

lora = lora_trainable_params()
full = full_params()
print(f"LoRA trainable params : {lora/1e6:8.2f} M")
print(f"~Full model params    : {full/1e9:8.2f} B")
print(f"Trainable fraction    : {100*lora/full:8.3f} %")

# Effective batch size drives training stability.
per_device, grad_accum, n_gpus = 4, 8, 1
print(f"\nEffective batch size  : {per_device * grad_accum * n_gpus} sequences")
print("(raise grad_accum to grow the effective batch without more VRAM)")


## Production Deployment

### Running fine-tuning as a job, then serving the result

Fine-tuning is a **batch job**, not a service: package it as a container and run it on a GPU node (a Kubernetes `Job`, a managed training job, or a Slurm/Ray task). Serving the *result* is a separate online concern (see the Inference notebook) — typically vLLM/TGI on the merged weights, or the base model with the LoRA adapter loaded at runtime.

#### Docker image for the training job

```dockerfile
FROM nvcr.io/nvidia/pytorch:24.05-py3      # CUDA + PyTorch base

WORKDIR /workspace
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt   # trl, peft, datasets, bitsandbytes, accelerate

COPY train_sft.py .

# Secrets (HF token, registry creds) come from the orchestrator, never baked in.
ENV HF_HOME=/cache/hf \
    TRANSFORMERS_VERBOSITY=info

# The script reads config from env/args and writes the adapter to $OUTPUT_DIR.
ENTRYPOINT ["python", "train_sft.py"]
```

#### Kubernetes Job for the fine-tuning run

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: sft-llama-domain
spec:
  backoffLimit: 1                       # don't silently retry an expensive GPU job
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: trainer
          image: registry.example.com/sft-trainer:v3
          args: ["--base", "meta-llama/Llama-3.1-8B-Instruct",
                 "--dataset", "s3://data/support-sft/v5",
                 "--method", "qlora", "--epochs", "2", "--lr", "2e-4",
                 "--output", "s3://models/support-sft/v3"]
          resources:
            limits:
              nvidia.com/gpu: 1
              memory: 48Gi
            requests:
              cpu: "8"
              memory: 32Gi
          env:
            - name: HF_TOKEN
              valueFrom: { secretKeyRef: { name: hf-creds, key: token } }
          volumeMounts:
            - { name: cache, mountPath: /cache/hf }
      volumes:
        - name: cache
          emptyDir: { sizeLimit: 100Gi }
```

After the job writes the adapter (or merged model) to object storage and it passes evaluation, promote it through the model registry and roll it out to the serving stack with canary/shadow traffic — exactly like any model version change.

## Monitoring and Observability

### Monitoring a fine-tuning run (and the model it produces)

#### Key Metrics to Track

- **Training & validation loss** — both should fall; a rising *val* loss while *train* loss falls is overfitting → stop.
- **Learning-rate schedule & grad norm** — confirm warmup/decay are applied; exploding or NaN grad norm signals LR too high or bad data.
- **Throughput (tokens/sec, samples/sec) and GPU utilization/memory** — efficiency signals; low GPU util means a data or padding bottleneck (enable packing).
- **Task-level eval metrics** — exact-match, schema-validity rate, win-rate vs the base model, or a held-out benchmark; the only metrics that capture *quality*.
- **Catastrophic-forgetting probes** — a small fixed general-ability suite run before/after to catch regressions on capabilities you didn't fine-tune.
- **Post-deploy quality & drift** — for the served model, track output quality, schema-validity, and input drift over time.

#### Logging Best Practices

- **Use an experiment tracker** (Weights & Biases, MLflow, TensorBoard) to log loss curves, hyperparameters, dataset hash, and sample generations per run.
- **Log sample generations on the eval set** every N steps — curves don't show that the model started emitting empty or runaway outputs.
- **Record the full recipe** — base model, dataset version/hash, template, and all hyperparameters — so any checkpoint is reproducible.
- **Structure run metadata as JSON** keyed by run id and model version for queryable comparison across experiments.

## Troubleshooting

#### Issue 1: Loss goes down but the model ignores instructions / repeats the prompt

**Symptoms:** validation loss looks fine, yet generations echo the question or ramble off-task.

**Cause:** the prompt wasn't masked (loss computed over the instruction), or a wrong chat template was used at train or serve time.

**Solution:** mask prompt tokens to `-100`, render with `apply_chat_template`, and use the *same* template when serving. Verify a rendered training example by eye.

#### Issue 2: The model never stops generating

**Symptoms:** outputs run to the max token limit with trailing garbage.

**Cause:** the EOS / end-of-turn token wasn't present (or was masked) in the training targets, so the model never learned to stop.

**Solution:** confirm the template appends EOS to the response and that it is a *learned* (unmasked) token; set the correct stop token at inference.

#### Issue 3: CUDA out-of-memory during training

**Symptoms:** OOM at model load or partway through the first steps.

**Cause:** full fine-tuning or large batch/sequence length exceeds VRAM; optimizer states for full FT are ~2–3x the model size.

**Solution:** switch to QLoRA (4-bit base), enable gradient checkpointing, lower `per_device_batch_size` and raise `gradient_accumulation_steps`, shrink `max_seq_length`, or use DeepSpeed ZeRO offload.

#### Issue 4: Fine-tuned model lost general ability (catastrophic forgetting)

**Symptoms:** great on the new task, noticeably worse on everything else.

**Cause:** too many epochs / too high LR on a narrow dataset overwrote general capabilities.

**Solution:** fewer epochs and lower LR, mix in general instruction data, prefer LoRA with a modest rank, and run a general-ability probe set as a gate.

## Comparison with Alternatives

### How instruction fine-tuning compares to other adaptation methods

| Dimension | Instruction FT (full) | LoRA / QLoRA | Prompting / Few-shot | RAG | RLHF / DPO |
|-----------|-----------------------|--------------|----------------------|-----|------------|
| What it changes | All weights | Small adapters | Nothing (context only) | Nothing (adds retrieval) | Weights, on *preferences* |
| Best for | Deep behavior/format change | Cheap behavior/format change | Quick tasks, few examples | Injecting fresh/private facts | Aligning helpfulness/tone |
| Data needed | 1k–100k+ pairs | Hundreds–thousands of pairs | 0–handful | A document corpus | Preference pairs |
| Cost / hardware | High (multi-GPU) | Low (1 GPU) | None (inference only) | Low–moderate | High (pairs + training) |
| Inference cost | Same as base | Base + tiny adapter | Higher (long prompts) | Higher (retrieval + context) | Same as base |
| Updates facts? | Poorly (frozen at train) | Poorly | Yes (in prompt) | **Yes (live)** | No |

### When to Choose This Approach

Choose **instruction fine-tuning** when you need *consistent behavior, format, or style* that prompting can't reliably enforce and you have examples to teach it — and start with **LoRA/QLoRA** unless you've measured a clear need for full fine-tuning. Use **RAG instead** when the gap is missing *facts*, **prompting** when examples are scarce or the task changes constantly, and add **DPO/RLHF after** SFT when you need to optimize subjective qualities like helpfulness and tone.

## Resources

### Official Documentation

- TRL (`SFTTrainer`): https://huggingface.co/docs/trl/sft_trainer
- PEFT (LoRA/QLoRA): https://huggingface.co/docs/peft/index
- Transformers `Trainer`: https://huggingface.co/docs/transformers/main_classes/trainer
- bitsandbytes (4-bit quantization): https://huggingface.co/docs/bitsandbytes/index
- Chat templates: https://huggingface.co/docs/transformers/chat_templating

### Tutorials and Guides

- Hugging Face alignment handbook (SFT + DPO recipes): https://github.com/huggingface/alignment-handbook
- QLoRA paper (4-bit fine-tuning): https://arxiv.org/abs/2305.14314
- LoRA paper (low-rank adaptation): https://arxiv.org/abs/2106.09685
- "Training language models to follow instructions" (InstructGPT): https://arxiv.org/abs/2203.02155

### Community Resources

- Hugging Face forums: https://discuss.huggingface.co/
- TRL GitHub Discussions: https://github.com/huggingface/trl/discussions
- Stack Overflow tags: `huggingface-transformers`, `peft`, `large-language-model`

### Related Technologies

- **Frameworks:** Hugging Face TRL/PEFT, Axolotl, Unsloth, Llama-Factory, NeMo
- **Preference optimization:** DPO, ORPO, KTO, RLHF/PPO
- **Serving the result:** vLLM, Text Generation Inference (TGI), Triton
- **Data & eval:** Argilla, distilabel, lm-evaluation-harness